# PKR Vision — registered classification driver
This notebook orchestrates checked-in package code and immutable configs. Use a GPU runtime, a pinned commit, and persistent Drive storage.

In [ ]:
REPOSITORY_URL = "REPLACE_WITH_PUBLIC_REPOSITORY_URL"
COMMIT = "REPLACE_WITH_COMMIT_SHA"
assert "REPLACE_WITH" not in REPOSITORY_URL and "REPLACE_WITH" not in COMMIT

In [ ]:
!git clone --filter=blob:none $REPOSITORY_URL /content/pkr-vision
%cd /content/pkr-vision
!git checkout $COMMIT
import sys, shutil
print(sys.version)
!python -m pip install --upgrade pip
!python -m pip install -e '.[train]'
if shutil.which('nvidia-smi'):
    !nvidia-smi
else:
    raise RuntimeError('No NVIDIA GPU runtime detected. In Colab, choose Runtime > Change runtime type > GPU, then reconnect and rerun from the top.')


Upload `classification-v1.tar.gz` to `MyDrive/Datasets/CurrencyDetection` or change the next cell to according to your location. The next cell verifies the exact archive and manifest before exposing them to training.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, platform, subprocess, sys, tarfile
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/pkr-vision')
ARCHIVE = DRIVE_ROOT / 'classification-v1.tar.gz'
DATA_ROOT = DRIVE_ROOT / 'classification-v1'
EXPECTED_ARCHIVE = 'a6ccf9a55a43bbc638f25e4ef074dc775d7bf202c4ad2b3a341d85892c8d68ba'
EXPECTED_MANIFEST = 'd36533b491482a3eb4b84cdf6739ab346ac6f5d619fc59e088eca4d0e0e08413'
assert ARCHIVE.is_file(), f'Upload {ARCHIVE.name} to {ARCHIVE.parent}'
assert hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() == EXPECTED_ARCHIVE
if not DATA_ROOT.is_dir():
    with tarfile.open(ARCHIVE, 'r:gz') as bundle:
        bundle.extractall(DRIVE_ROOT, filter='data')
assert hashlib.sha256((DATA_ROOT / 'samples.jsonl').read_bytes()).hexdigest() == EXPECTED_MANIFEST
Path('data/processed/classification-v1').symlink_to(DATA_ROOT, target_is_directory=True)
RUN_ROOT = DRIVE_ROOT / 'runs' / COMMIT
RUN_ROOT.mkdir(parents=True, exist_ok=True)
Path('artifacts/mlruns').symlink_to(RUN_ROOT, target_is_directory=True)
print(json.dumps({'python': sys.version, 'platform': platform.platform(), 'commit': COMMIT, 'manifest': EXPECTED_MANIFEST}, indent=2))

In [ ]:
!pkrvision plan-classification-ablation data/processed/classification-v1/samples.jsonl artifacts/mlruns/classification/plan.json
!python -c "import json; print([(x['requested_per_class'], x['status'], x['reason']) for x in json.load(open('artifacts/mlruns/classification/plan.json'))])"

Run all eight feasible conditions. Each writes its checkpoint, validation/test predictions, history, hashes, calibration, and metrics directly to Drive.

In [ ]:
from pathlib import Path
for level in ('10', '50', '100', 'full'):
    for augmented in (False, True):
        suffix = 'aug' if augmented else 'noaug'
        run_dir = Path(f'artifacts/mlruns/classification/n{level}-{suffix}')
        run_json = run_dir / 'run.json'
        if run_json.is_file():
            print(f'SKIP completed run: n{level}-{suffix}')
            continue
        command = ['pkrvision', 'train-classifier', 'configs/research/ablation.yaml', 'data/processed/classification-v1/internal', 'artifacts/mlruns/classification/plan.json', level, str(run_dir)]
        if augmented:
            command.append('--augmentation')
        subprocess.run(command, check=True)
!pkrvision audit-classification-runs artifacts/mlruns/classification/plan.json artifacts/mlruns/classification artifacts/mlruns/classification/run-matrix.json
!python -c "import json; print(json.load(open('artifacts/mlruns/classification/run-matrix.json'))['counts'])"


In [ ]:
!pkrvision select-classifier artifacts/mlruns/classification/nfull-noaug/run.json artifacts/mlruns/classification/nfull-aug/run.json artifacts/mlruns/classification/selection.json
with open('artifacts/mlruns/classification/selection.json') as selection_file:
    selection = json.load(selection_file)
selected_suffix = 'aug' if selection['selected'] == 'registered_augmentation' else 'noaug'
checkpoint = f'artifacts/mlruns/classification/nfull-{selected_suffix}/best.pt'
!pkrvision evaluate-external $checkpoint data/processed/classification-v1/external artifacts/mlruns/classification/external/result.json
!pkrvision export-classifier $checkpoint artifacts/mlruns/classification/release/classifier.onnx
!pkrvision validate-classifier-onnx $checkpoint artifacts/mlruns/classification/release/classifier.onnx data/processed/classification-v1/internal/test artifacts/mlruns/classification/release/onnx-validation.json
predictions = f'artifacts/mlruns/classification/nfull-{selected_suffix}/test-predictions.npz'
!pkrvision generate-gradcam $checkpoint data/processed/classification-v1/internal/test $predictions artifacts/mlruns/classification/release/gradcam

Do not publish metrics directly from notebook output. Download the complete commit-scoped run directory, run ONNX agreement and qualitative failure analysis, validate canonical result JSON, and build the quality-gated release manifest.